In [1]:
# Official packages
import os
import copy
import math
import re
import time

from pprint import pprint
from itertools import chain

# Third-party packages
import numpy as np
import scipy

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm


import pyopencl as cl
#for item in cl.get_platforms():
#    print(item.get_devices())

import pynufft
device_list = pynufft.helper.device_list()
for item in device_list:
    print(item)
    
# In-house packages
from BrukerPV360 import BrukerPV360Exp


def normalize_array(arr:np.array):
    return arr/np.max(np.abs(arr))

C:\Anaconda3\lib\site-packages\pytools\persistent_dict.py:59: UserWarning: pytools.persistent_dict: unable to import 'siphash24.siphash13', falling back to hashlib.sha256
  warn("pytools.persistent_dict: unable to import 'siphash24.siphash13', "


No cuda device found. Check your pycuda installation.
('ocl', 0, 0, <pyopencl.Platform 'Intel(R) OpenCL Graphics' at 0x1e116875720>, <pyopencl.Device 'Intel(R) UHD Graphics' on 'Intel(R) OpenCL Graphics' at 0x1e115363ac0>, <reikna.cluda.ocl.Thread object at 0x000001E11915A3D0>, 64)
('ocl', 1, 0, <pyopencl.Platform 'Intel(R) OpenCL' at 0x1e11984eb90>, <pyopencl.Device '11th Gen Intel(R) Core(TM) i7-11850H @ 2.50GHz' on 'Intel(R) OpenCL' at 0x1e116f19bf0>, <reikna.cluda.ocl.Thread object at 0x000001E11914DFD0>, 1)
('ocl', 2, 0, <pyopencl.Platform 'Intel(R) FPGA Emulation Platform for OpenCL(TM)' at 0x1e11984f050>, <pyopencl.Device 'Intel(R) FPGA Emulation Device' on 'Intel(R) FPGA Emulation Platform for OpenCL(TM)' at 0x1e116f1a370>, <reikna.cluda.ocl.Thread object at 0x000001E11911F940>, 64)


# hpSpiral3d Recon

We first build the k-mesh as in the previous section

In [2]:
raw_path = "C:/Users/Xiao JI/OneDrive - University of California, San Francisco/Documents/Projects/HP-bSSFP/20240731-FreqSweep/proton-pm-2kHz/7"

hpSpiral3d = BrukerPV360Exp(raw_path, is_verbose=False)

FID = hpSpiral3d.dataset['DATA']['rawdata'][0]

plt.plot(np.real(FID))
plt.plot(np.imag(FID))
plt.plot(np.abs(FID))
plt.show

spiral_y = (hpSpiral3d.dataset['PARAM']['method'][0]['PVM_SpiralShape1'])
spiral_x = (hpSpiral3d.dataset['PARAM']['method'][0]['PVM_SpiralShape2'])

traj_cal_x = np.cumsum(spiral_x)[::5]
traj_cal_y = np.cumsum(spiral_y)[::5]

shift_cos = (hpSpiral3d.dataset['PARAM']['method'][0]['PVM_SpiralInterleavCos'])
shift_sin = (hpSpiral3d.dataset['PARAM']['method'][0]['PVM_SpiralInterleavSin'])
shift_angle = np.angle(shift_cos+1j*shift_sin)

traj_cal_seg = traj_cal_x + 1j*traj_cal_y
traj_cal = []
for angle in shift_angle:
    traj_cal.append(traj_cal_seg[:155] * np.exp( angle*1j))

traj_cal = np.asarray(traj_cal).flatten()
traj_Kxy = traj_cal / np.max(np.abs(traj_cal))

plt.figure()
plt.plot(traj_Kxy.real, traj_Kxy.imag, c= "black", marker='.', linestyle=':')
plt.scatter(traj_Kxy.real, traj_Kxy.imag, c=(range(len(traj_cal))), s=40, cmap='bwr')
plt.show()

traj_Kz = hpSpiral3d.dataset['PARAM']['method'][0]['PVM_EncSteps2']
traj_Kz = traj_Kz / np.max(traj_Kz)

traj_K3d = []
for kz in traj_Kz:
    for kxy in traj_Kxy:
        traj_K3d.append([kxy.real, kxy.imag, kz])
traj_K3d = np.asarray(traj_K3d)

traj_K3d_norm = np.linalg.norm(traj_K3d, ord=2, axis=1)
plt.figure()
plt.plot(traj_K3d_norm, c= "black", marker=' ',alpha=.25)
plt.show()

fig = plt.figure(figsize = (10, 7))
ax = plt.axes(projection ="3d")
ax.scatter3D(traj_K3d[:,0], traj_K3d[:,1], traj_K3d[:,2], c = traj_K3d_norm, cmap='bwr')
plt.title("simple 3D noscatter plot") 
# show plot
plt.show()

IndexError: list index out of range

The stupid Bruker recon has bug

In [ ]:
bruker_data = hpSpiral3d.dataset['DATA']['2dseq']

img_shape = [int(x) for x in hpSpiral3d.dataset['PARAM']['method'][0]['PVM_Matrix']]
print(img_shape)
NR = hpSpiral3d.dataset['PARAM']['method'][0]['PVM_NRepetitions']

bruker_data = np.reshape(bruker_data, img_shape)
plt.figure()
plt.imshow(bruker_data[8], cmap=cm.YlGnBu)
plt.show()

Our 3d NuFFT recon with 4090 Video card is pretty fast

In [ ]:
Nd = tuple(img_shape)  # image size
Kd = tuple([8*x for x in img_shape])  # k-space size
Jd = (6, 6, 6)  # interpolation size

print(np.shape(FID), np.shape(traj_Kxy), np.shape(traj_K3d))
print(Nd, Kd, Jd)

# this line takes the CPU approach, which is at least 4 times slower with all 16 cores 100% occupied
spiralStackObj = pynufft.NUFFT() 
# this line takes the GPU approach, which requires the pycuda package finds a GPU
#spiralStackObj = pynufft.NUFFT(pynufft.helper.device_list()[0])
spiralStackObj.plan(traj_K3d*np.pi, Nd, Kd, Jd)

time1 = time.time()
recon = spiralStackObj.solve(FID, solver='cg', maxiter=500)
time2 = time.time()
print(time2-time1)



In [ ]:
from itertools import product
from matplotlib import pyplot as plt
N = 16
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(projection="3d")
space = np.array([*product(range(N), range(N), range(N))]) # all possible triplets of numbers from 0 to N-1
volume = normalize_array(np.abs(recon))
ax.scatter(space[:,0], space[:,1], space[:,2], c=volume, s=volume*20, cmap='YlGnBu')

In [ ]:
plt.figure()
plt.imshow(np.abs(recon)[:,:,8], cmap=cm.YlGnBu)
plt.show()
plt.figure()
plt.imshow(np.abs(recon)[:,8,:], cmap=cm.YlGnBu)
plt.show()
plt.figure()
plt.imshow(np.abs(recon)[8,:,:], cmap=cm.YlGnBu)
plt.show()

And the NuFFT result seems resonable.